# Lagrangian Trajectory Movies & Truncated PSDs

Load saved trajectory data from notebook 06, generate per-trajectory movies
showing Bz with the particle position and tail, then manually set end times
to exclude unphysical regions and recompute frequency PSDs.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pathlib import Path

from reconn_wave_power.io import read_simulation
from reconn_wave_power.processing import compute_flux_function
from reconn_wave_power.spectrum import compute_psd_time

%matplotlib inline

## Configuration

In [ ]:
INPUT_FILE = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/input/input"
OUTPUT_FOLDER = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/Output"
BATCH_FILE = "lagrangian_batch.nc"

MOVIE_DIR = os.path.join(os.path.dirname(os.getcwd()), "movies") if os.path.basename(os.getcwd()) == "notebooks" else "movies"
os.makedirs(MOVIE_DIR, exist_ok=True)

FRAME_STRIDE = 10       # render every Nth simulation frame (reduces 1001 → ~100)
TAIL_LEN = 20           # number of previous *strided* frames shown as tail
MOVIE_FPS = 20
MOVIE_DPI = 150
PSD_METHOD = "fft"
N_CONTOURS = 20         # number of flux-function contour levels (field lines)

## Load Data

In [ ]:
# Saved trajectory + sampled field data
ds_batch = xr.open_dataset(BATCH_FILE)
print(ds_batch)

x_traj = ds_batch["x_traj"].values        # (N_total, n_times)
y_traj = ds_batch["y_traj"].values
active_mask = ds_batch["active_mask"].values.astype(bool)
t_traj = ds_batch["time"].values
x0s = ds_batch["x0"].values
y0s = ds_batch["y0"].values
N_total = len(x0s)
dt = ds_batch.attrs["dt"]
n_lines = ds_batch.attrs["n_lines"]
n_per_line = ds_batch.attrs["n_per_line"]

print(f"\nTrajectories: {N_total}, Timesteps: {len(t_traj)}, dt: {dt}")

In [ ]:
# Load simulation Bz for movie frames
ds = read_simulation(
    input_file=INPUT_FILE,
    output_folder=OUTPUT_FOLDER,
    fields=("B",),
    progress=True,
)
print(ds)

## Compute Global Color Limits

In [ ]:
#bz_min = float(ds["Bz"].min())
#bz_max = float(ds["Bz"].max())
bz_min = 0
bz_max = .5
vlim = max(abs(bz_min), abs(bz_max))
print(f"Bz range: [{bz_min:.4f}, {bz_max:.4f}], symmetric clim: +/-{vlim:.4f}")

## Preload Bz Frames

Batch-load strided Bz frames into memory so each timestep is read exactly once.

In [ ]:
import dask

nt = len(t_traj)
frame_indices = np.arange(0, nt, FRAME_STRIDE)
n_frames = len(frame_indices)
print(f"Strided frames: {n_frames} (stride={FRAME_STRIDE}, original={nt})")

# Batch-load all strided Bz and Bx frames via dask.compute in chunks of 64
BATCH_SIZE = 64
bz_frames = np.empty((n_frames, len(ds.x), len(ds.y)), dtype=np.float32)
bx_frames = np.empty((n_frames, len(ds.x), len(ds.y)), dtype=np.float32)

for batch_start in range(0, n_frames, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n_frames)
    delayed_bz = [ds["Bz"].isel(time=int(frame_indices[i])).data
                  for i in range(batch_start, batch_end)]
    delayed_bx = [ds["Bx"].isel(time=int(frame_indices[i])).data
                  for i in range(batch_start, batch_end)]
    results = dask.compute(*(delayed_bz + delayed_bx))
    n_batch = batch_end - batch_start
    for j in range(n_batch):
        bz_frames[batch_start + j] = results[j]
        bx_frames[batch_start + j] = results[n_batch + j]
    print(f"  Loaded frames {batch_start}–{batch_end - 1} / {n_frames - 1}")

print(f"bz_frames shape: {bz_frames.shape}, size: {bz_frames.nbytes / 1e6:.1f} MB")
print(f"bx_frames shape: {bx_frames.shape}, size: {bx_frames.nbytes / 1e6:.1f} MB")

# Compute flux function ψ for each frame
y_coords = ds.y.values
y_axis = 1  # bx_frames has shape (n_frames, nx, ny), y is axis 1
from scipy.integrate import cumulative_trapezoid
psi_frames = cumulative_trapezoid(bx_frames, y_coords, axis=2, initial=0)
print(f"psi_frames shape: {psi_frames.shape}, size: {psi_frames.nbytes / 1e6:.1f} MB")

## Generate Movies

In [ ]:
for traj_idx in range(N_total):
    x0 = x0s[traj_idx]
    y0 = y0s[traj_idx]
    fname = os.path.join(
        MOVIE_DIR,
        f"trajectory_{traj_idx:02d}_x0_{x0:.1f}_y0_{y0:.1f}.mp4",
    )

    fig, ax = plt.subplots(figsize=(12, 4))
    im = ax.pcolormesh(
        ds.x, ds.y, bz_frames[0].T, shading="auto", cmap="RdBu_r",
        vmin=-vlim, vmax=vlim,
    )
    # Initial flux-function contours (field lines)
    contour_set = ax.contour(
        ds.x.values, ds.y.values, psi_frames[0].T,
        levels=N_CONTOURS, colors="k", linewidths=0.5,
    )
    (tail_line,) = ax.plot([], [], "-", color="black", lw=1.5, alpha=0.7)
    (marker,) = ax.plot([], [], "o", color="black", ms=6, mec="white", mew=0.8)
    ax.set_xlabel(r"x [$d_i$]")
    ax.set_ylabel(r"y [$d_i$]")
    ax.set_aspect("equal")
    plt.colorbar(im, ax=ax, label="Bz")
    title = ax.set_title("")

    xt = x_traj[traj_idx]
    yt = y_traj[traj_idx]

    def update(frame_idx, xt=xt, yt=yt):
        nonlocal contour_set
        im.set_array(bz_frames[frame_idx].T.ravel())
        # Update flux-function contours: remove old, draw new
        for c in contour_set.collections:
            c.remove()
        contour_set = ax.contour(
            ds.x.values, ds.y.values, psi_frames[frame_idx].T,
            levels=N_CONTOURS, colors="k", linewidths=0.5,
        )
        # Map strided frame index back to original time index for trajectory coords
        orig_idx = frame_indices[frame_idx]
        # Tail: look back TAIL_LEN strided frames
        t_start = max(0, frame_idx - TAIL_LEN)
        tail_orig = frame_indices[t_start:frame_idx + 1]
        tail_line.set_data(xt[tail_orig], yt[tail_orig])
        # Current position
        marker.set_data([xt[orig_idx]], [yt[orig_idx]])
        title.set_text(
            f"Trajectory {traj_idx:02d}  "
            f"(x0={x0:.1f}, y0={y0:.1f})  "
            f"t = {t_traj[orig_idx]:.1f} $\\Omega_{{ci}}^{{-1}}$"
        )
        return im, tail_line, marker, title

    anim = FuncAnimation(fig, update, frames=n_frames, interval=50, blit=False)
    plt.close(fig)
    anim.save(fname, writer="ffmpeg", fps=MOVIE_FPS, dpi=MOVIE_DPI)
    print(f"[{traj_idx + 1}/{N_total}] Saved: {fname}")

print("\nAll movies saved.")

## Define Per-Trajectory End Times

After reviewing the movies, set the end time index for each trajectory.
Trajectories that stay physical for the full run keep the default (last index).
Edit entries below for trajectories that enter unphysical regions.

In [ ]:
# Default: use all timesteps
end_time_idx = np.full(N_total, len(t_traj), dtype=int)

# --- Manually override unphysical trajectories ---
# end_time_idx[3] = 500
# end_time_idx[17] = 750

# Summary table
print(f"{'Traj':>4s}  {'x0':>8s}  {'y0':>8s}  {'end_idx':>7s}  {'end_time':>10s}")
print("-" * 45)
for i in range(N_total):
    t_end = t_traj[end_time_idx[i] - 1] if end_time_idx[i] > 0 else 0.0
    flag = "  <-- truncated" if end_time_idx[i] < len(t_traj) else ""
    print(f"{i:4d}  {x0s[i]:8.1f}  {y0s[i]:8.1f}  {end_time_idx[i]:7d}  {t_end:10.1f}{flag}")

## Compute Truncated PSDs

In [ ]:
COMPONENTS = ["Ex", "Ey", "Ez", "Bx", "By", "Bz"]
psd_results = {}
freq = None

for comp in COMPONENTS:
    print(f"Computing PSDs for {comp}...")
    sampled = ds_batch[f"{comp}_sampled"].values  # (N_total, n_times)
    psd_list = []
    for i in range(N_total):
        end = end_time_idx[i]
        vals = sampled[i, :end]
        t_valid = t_traj[:end]
        da = xr.DataArray(vals, dims=("time",), coords={"time": t_valid})
        f, Pxx = compute_psd_time(da, dt=dt, method=PSD_METHOD)
        psd_list.append(Pxx)
        if freq is None:
            freq = f
    psd_results[comp] = np.array(psd_list)

print(f"\nFrequency array: {len(freq)} points, [{freq[0]:.4f}, {freq[-1]:.4f}]")
print(f"PSD shape per component: {psd_results[COMPONENTS[0]].shape}")

## Plot PSDs

In [ ]:
FIELD = "Bz"

# Unique x-line starting positions
x_lines = np.unique(x0s)
colors = plt.cm.tab10(np.linspace(0, 1, len(x_lines)))

fig, ax = plt.subplots(figsize=(10, 6))
for i_line, xl in enumerate(x_lines):
    mask = x0s == xl
    idxs = np.where(mask)[0]
    for j in idxs:
        ax.semilogy(freq, psd_results[FIELD][j], color=colors[i_line],
                    alpha=0.5, lw=0.8)
    ax.semilogy([], [], color=colors[i_line], lw=2, label=f"x0 = {xl:.1f}")

ax.set_xlabel(r"frequency [$\Omega_{ci}$]")
ax.set_ylabel("PSD")
ax.set_title(f"Truncated Lagrangian PSD of {FIELD}")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Save Truncated Results

In [ ]:
ds_out = xr.Dataset(
    {
        "end_time_idx": (["trajectory"], end_time_idx),
        "x0": (["trajectory"], x0s),
        "y0": (["trajectory"], y0s),
    },
    coords={
        "trajectory": np.arange(N_total),
        "frequency": freq,
    },
    attrs={
        "source_file": BATCH_FILE,
        "psd_method": PSD_METHOD,
        "dt": dt,
    },
)

for comp in COMPONENTS:
    ds_out[f"psd_{comp}"] = (["trajectory", "frequency"], psd_results[comp])

out_path = Path(".") / "lagrangian_batch_truncated.nc"
ds_out.to_netcdf(out_path)
print(f"Saved to {out_path}")
print(ds_out)